# 🔧 Provider Compatibility Layer Configuration Manager

This notebook manages the JSON file used by the **provider compatibility layer**.

It stores compatibility rules in the same secrets area used by your other notebooks:

`~/.secrets/provider_compatibility_configs.json`

## Purpose

This file lets your notebooks know which request fields should be removed or ignored
for specific providers before making an OpenAI-style API call.

### Starting point
- **`Not Specified`** → do nothing
- **`gemini`** → remove known unsupported OpenAI chat fields

## Initial Gemini compatibility
The first version preloads these Gemini exclusions:

- `frequency_penalty`
- `seed`

You can add more later as you learn what each provider accepts.


In [ ]:

import json
from pathlib import Path
from IPython.display import display, Markdown

CONFIG_FILE = Path.home() / ".secrets" / "provider_compatibility_configs.json"
CONFIG_FILE.parent.mkdir(exist_ok=True)

DEFAULT_KEY = "_default"
DEFAULT_PROVIDER_NAME = "Not Specified"

STARTER_CONFIGS = {
    DEFAULT_KEY: DEFAULT_PROVIDER_NAME,
    DEFAULT_PROVIDER_NAME: {
        "description": "Default compatibility behavior. No parameters are removed.",
        "drop_fields": []
    },
    "gemini": {
        "description": "Google Gemini OpenAI-compatible endpoint. Remove currently known unsupported fields.",
        "drop_fields": ["frequency_penalty", "seed"]
    }
}


def normalize_configs(configs):
    """Ensure expected starter entries exist without overwriting user customizations."""
    if not isinstance(configs, dict):
        configs = {}

    if DEFAULT_KEY not in configs:
        configs[DEFAULT_KEY] = DEFAULT_PROVIDER_NAME

    for name, payload in STARTER_CONFIGS.items():
        if name == DEFAULT_KEY:
            continue
        if name not in configs or not isinstance(configs[name], dict):
            configs[name] = payload.copy()
        else:
            configs[name].setdefault("description", payload.get("description", ""))
            configs[name].setdefault("drop_fields", payload.get("drop_fields", []).copy())

    return configs


def load_configs():
    if CONFIG_FILE.exists():
        with CONFIG_FILE.open() as f:
            data = json.load(f)
    else:
        data = {}
    return normalize_configs(data)


def save_configs(configs):
    configs = normalize_configs(configs)
    with CONFIG_FILE.open("w") as f:
        json.dump(configs, f, indent=2)


def get_provider_names(configs):
    return sorted([k for k in configs.keys() if k != DEFAULT_KEY])


def list_configs():
    configs = load_configs()
    names = get_provider_names(configs)

    default_name = configs.get(DEFAULT_KEY, DEFAULT_PROVIDER_NAME)
    display(Markdown(f"### 📦 Provider Compatibility Configurations (default: `{default_name}`)"))
    display(Markdown(f"**Storage:** `{CONFIG_FILE}`"))

    for name in names:
        cfg = configs.get(name, {})
        star = " ⭐" if name == default_name else ""
        drop_fields = cfg.get("drop_fields", [])
        description = cfg.get("description", "")
        display(Markdown(
            f"- `{name}`{star}  \n"
            f"  description: {description or '_none_'}  \n"
            f"  drop_fields: `{drop_fields}`"
        ))


def create_or_update_config():
    configs = load_configs()

    name = input(f"🔖 Enter provider name [{DEFAULT_PROVIDER_NAME}, gemini, anthropic, etc.]: ").strip()
    if not name or name == DEFAULT_KEY:
        display(Markdown(f"❌ **Invalid name. `{DEFAULT_KEY}` is reserved.**"))
        return

    existing = configs.get(name, {})
    description = input(f"📝 Description [{existing.get('description','')}]: ").strip() or existing.get("description", "")
    current_drop_fields = ", ".join(existing.get("drop_fields", []))
    raw_fields = input(f"🧹 Fields to drop (comma-separated) [{current_drop_fields}]: ").strip()

    if raw_fields:
        drop_fields = [field.strip() for field in raw_fields.split(",") if field.strip()]
    else:
        drop_fields = existing.get("drop_fields", [])

    configs[name] = {
        "description": description,
        "drop_fields": drop_fields
    }

    make_default = input("⭐ Set this provider as default compatibility config? (y/N): ").strip().lower() == "y"
    if make_default:
        configs[DEFAULT_KEY] = name

    save_configs(configs)
    display(Markdown(f"✅ **Provider compatibility config `{name}` saved.**"))
    if make_default:
        display(Markdown(f"🌟 **Default compatibility config set to `{name}`.**"))


def set_default_config():
    configs = load_configs()
    names = get_provider_names(configs)
    if not names:
        display(Markdown("❌ **No provider configs found.**"))
        return

    list_configs()
    name = input("⭐ Enter provider name to set as default: ").strip()
    if name not in configs or name == DEFAULT_KEY:
        display(Markdown(f"❌ **Provider config `{name}` not found.**"))
        return

    configs[DEFAULT_KEY] = name
    save_configs(configs)
    display(Markdown(f"🌟 **Default compatibility config set to `{name}`.**"))


def delete_config():
    configs = load_configs()
    names = get_provider_names(configs)
    if not names:
        display(Markdown("❌ **No provider configs to delete.**"))
        return

    list_configs()
    name = input("🗑️ Enter provider name to delete: ").strip()
    if name not in configs or name == DEFAULT_KEY:
        display(Markdown(f"❌ **Provider config `{name}` not found.**"))
        return

    if name == DEFAULT_PROVIDER_NAME:
        display(Markdown(f"❌ **`{DEFAULT_PROVIDER_NAME}` is a protected starter config and cannot be deleted.**"))
        return

    confirm = input(f"⚠️ Really delete `{name}`? (y/N): ").strip().lower() == "y"
    if not confirm:
        display(Markdown("✅ **Delete cancelled.**"))
        return

    configs.pop(name, None)

    if configs.get(DEFAULT_KEY) == name:
        configs[DEFAULT_KEY] = DEFAULT_PROVIDER_NAME
        display(Markdown(f"ℹ️ **Default reset to `{DEFAULT_PROVIDER_NAME}`.**"))

    save_configs(configs)
    display(Markdown(f"🗑️ **Deleted `{name}`.**"))


def rename_config():
    configs = load_configs()
    names = get_provider_names(configs)
    if not names:
        display(Markdown("❌ **No provider configs to rename.**"))
        return

    list_configs()
    old = input("✏️ Enter provider name to rename: ").strip()
    if old not in configs or old == DEFAULT_KEY:
        display(Markdown(f"❌ **Provider config `{old}` not found.**"))
        return

    if old == DEFAULT_PROVIDER_NAME:
        display(Markdown(f"❌ **`{DEFAULT_PROVIDER_NAME}` is a protected starter config and cannot be renamed.**"))
        return

    new = input("✏️ Enter new provider name: ").strip()
    if not new or new == DEFAULT_KEY or new in configs:
        display(Markdown("❌ **Invalid new name (reserved, empty, or already exists).**"))
        return

    configs[new] = configs.pop(old)
    if configs.get(DEFAULT_KEY) == old:
        configs[DEFAULT_KEY] = new

    save_configs(configs)
    display(Markdown(f"✅ **Renamed `{old}` → `{new}`.**"))


def preview_sanitize_example():
    configs = load_configs()
    list_configs()

    provider = input(f"🧪 Enter provider name to preview sanitize behavior [{configs.get(DEFAULT_KEY, DEFAULT_PROVIDER_NAME)}]: ").strip() or configs.get(DEFAULT_KEY, DEFAULT_PROVIDER_NAME)
    if provider not in configs:
        display(Markdown(f"❌ **Provider config `{provider}` not found.**"))
        return

    sample_payload = {
        "model": "example-model",
        "messages": [{"role": "user", "content": "Hello"}],
        "temperature": 0.2,
        "frequency_penalty": 0,
        "seed": 42,
        "presence_penalty": 0
    }

    drop_fields = set(configs[provider].get("drop_fields", []))
    sanitized = {k: v for k, v in sample_payload.items() if k not in drop_fields}

    display(Markdown(f"### 🧪 Preview for `{provider}`"))
    print("Original payload:")
    print(json.dumps(sample_payload, indent=2))
    print("\nSanitized payload:")
    print(json.dumps(sanitized, indent=2))


def initialize_file():
    configs = load_configs()
    save_configs(configs)
    display(Markdown(f"✅ **Initialized compatibility config file at** `{CONFIG_FILE}`"))


def menu():
    while True:
        display(Markdown(
            "## 🔧 Provider Compatibility Manager\n"
            "1) Initialize / refresh starter file\n"
            "2) List provider configs\n"
            "3) Create/Update provider config\n"
            "4) Set default provider config\n"
            "5) Delete provider config\n"
            "6) Rename provider config\n"
            "7) Preview sanitize behavior\n"
            "8) Exit"
        ))
        choice = input("➡️ Enter choice: ").strip()

        if choice == "1":
            initialize_file()
        elif choice == "2":
            list_configs()
        elif choice == "3":
            create_or_update_config()
        elif choice == "4":
            set_default_config()
        elif choice == "5":
            delete_config()
        elif choice == "6":
            rename_config()
        elif choice == "7":
            preview_sanitize_example()
        elif choice == "8":
            display(Markdown("👋 Done."))
            break
        else:
            display(Markdown("❌ **Invalid choice.**"))


menu()
